# Subtitles Generator

Pipeline overview:
1. Load audio from `sample_data/`
2. Detect speech segments with **Silero VAD** (`vad_utils.py`)
3. Translate each segment with Sarvam STT (`saaras:v3`, mode=`translate`)
4. Write `outputs/subtitles.srt`

> **Local vs streaming VAD:** this recipe runs Silero offline to cut a file into
> subtitle cues. For live captions over a WebSocket, use Sarvam's built-in
> `endpointing="vad"` instead (see `examples/Realtime_Speech_Captioning`).


In [ ]:
%pip install -r requirements.txt


## Setup


In [ ]:
from __future__ import annotations

import io
import os
from pathlib import Path

from dotenv import load_dotenv
from pydub import AudioSegment
from sarvamai import SarvamAI

from vad_utils import detect_speech_segments, load_audio_mono

load_dotenv()

SARVAM_API_KEY = os.getenv("SARVAM_API_KEY")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "Set SARVAM_API_KEY in your environment or .env file before running."
    )

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)
SAMPLE_DIR = Path("sample_data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# --- VAD tuning knobs (see README) ---
SAMPLE_RATE = 16000
VAD_THRESHOLD = 0.5          # speech probability cutoff
SMOOTH_WINDOW = 5            # moving-average frames (1 = off)
MIN_SPEECH_MS = 250.0        # drop clicks / noise blips
MIN_SILENCE_MS = 300.0       # hangover before ending an utterance
COMBINE_DURATION = 8.0       # max merged cue length (seconds)
COMBINE_GAP = 1.0            # merge if gap between utterances <= this
PAD_MS = 150.0               # context padding around each cue


## VAD helpers

Helpers live in `vad_utils.py` so they can be unit-tested without an API key:

| Function | Role |
|---|---|
| `smooth_probs` | Moving-average over Silero frame scores |
| `get_utterances` | Threshold + hangover → `(start, end)` |
| `merge_segments` | Join nearby cues up to `COMBINE_DURATION` |
| `pad_segments` | Add leading/trailing context for STT |
| `detect_speech_segments` | End-to-end local VAD pipeline |

`detect_speech_segments` loads Silero from `torch.hub` on first call.


In [ ]:
# Re-export for interactive exploration; prefer detect_speech_segments below.
from vad_utils import (
    filter_short_segments,
    frame_duration_sec,
    get_utterances,
    get_vad_probs,
    merge_segments,
    pad_segments,
    smooth_probs,
)

print(f"Silero frame @ {SAMPLE_RATE} Hz ≈ {frame_duration_sec(SAMPLE_RATE)*1000:.1f} ms")


## Transcription + SRT


In [ ]:
def detect_segments(audio_file: Path) -> list[tuple[float, float]]:
    audio = load_audio_mono(audio_file, SAMPLE_RATE)
    return detect_speech_segments(
        audio,
        SAMPLE_RATE,
        threshold=VAD_THRESHOLD,
        smooth_window=SMOOTH_WINDOW,
        min_speech_ms=MIN_SPEECH_MS,
        min_silence_ms=MIN_SILENCE_MS,
        max_duration=COMBINE_DURATION,
        max_gap=COMBINE_GAP,
        pad_ms=PAD_MS,
    )


def transcribe_segment(audio: AudioSegment, start_sec: float, end_sec: float) -> str:
    segment = audio[int(start_sec * 1000) : int(end_sec * 1000)]
    buf = io.BytesIO()
    segment.export(buf, format="wav")
    buf.seek(0)
    response = client.speech_to_text.transcribe(
        file=("segment.wav", buf, "audio/wav"),
        model="saaras:v3",
        mode="translate",
        language_code="unknown",
    )
    if hasattr(response, "transcript"):
        return response.transcript or ""
    if isinstance(response, dict):
        return response.get("transcript", "") or ""
    return str(response)


def format_timestamp(seconds: float) -> str:
    ms = int(round((seconds % 1) * 1000))
    if ms == 1000:
        seconds = int(seconds) + 1
        ms = 0
    total = int(seconds)
    hours, rem = divmod(total, 3600)
    minutes, secs = divmod(rem, 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{ms:03d}"


def write_srt(results: list[dict], output_path: Path) -> None:
    with output_path.open("w", encoding="utf-8") as fh:
        for i, row in enumerate(results, start=1):
            fh.write(f"{i}\n")
            fh.write(
                f"{format_timestamp(row['start_time'])} --> {format_timestamp(row['end_time'])}\n"
            )
            fh.write(f"{row['transcript']}\n\n")


## Run


In [ ]:
AUDIO_PATH = SAMPLE_DIR / "clip.wav"
if not AUDIO_PATH.exists():
    raise FileNotFoundError(
        f"Missing {AUDIO_PATH}. Add an audio file under sample_data/ first."
    )

segments = detect_segments(AUDIO_PATH)
if not segments:
    raise RuntimeError(
        f"No speech segments detected in {AUDIO_PATH}. "
        "Try lowering VAD_THRESHOLD or MIN_SPEECH_MS."
    )

print(f"Detected {len(segments)} segment(s):")
for i, (start, end) in enumerate(segments, start=1):
    print(f"  {i:02d}. {start:6.2f}s → {end:6.2f}s  ({end - start:5.2f}s)")

audio = AudioSegment.from_file(AUDIO_PATH)
results: list[dict] = []
for start, end in segments:
    transcript = transcribe_segment(audio, start, end)
    if transcript:
        results.append({"start_time": start, "end_time": end, "transcript": transcript})
        print(f"[{start:.1f}-{end:.1f}] {transcript}")

if not results:
    raise RuntimeError("VAD found segments but STT returned empty transcripts.")

srt_path = OUTPUT_DIR / "subtitles.srt"
write_srt(results, srt_path)
print(f"Wrote {srt_path}")
